In [1]:
# orb_15min_retest_sp500.py
# S&P 500, last ~60 trading days (Yahoo 15m).
# ORB (Opening Range Breakout) with Retest/Continuation entries.
# Scale-out in thirds: TP1=1R, TP2=2R, TP3=2.5R (1/3 each), BE after +1R.
# Includes: Profit Factor in totals, optional Monte Carlo, optional slippage/fee sensitivity.

import warnings
warnings.filterwarnings("ignore")

import time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# =============================
# USER PARAMETERS
# =============================

# Data (Yahoo intraday <60m => last ~60d only)
INTERVAL = "15m"
PERIOD = "60d"
USE_PERIOD = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120            # minutes after breakout to wait for retest
RETEST_CONFIRM_CLOSE = True     # retest bar must close back through the level
BREAKOUT_CUSHION_PCT = 0.0005   # 0.05% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.40   # reject entry bar if opposite wick > 40% of bar range

# Multiple trades per day (increase trade count)
MAX_TRADES_PER_SESSION = 2      # up to 2 trades per day
ALLOW_BOTH_SIDES = True         # allow long and short in same session if both trigger

# Scale-out in thirds (your request)
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]     # 1R, 2R, 2.5R
SCALE_SPLIT = (1/3, 1/3, 1/3)   # thirds
MOVE_STOP_TO_BE_AT_R = 1.0      # move stop to entry after +1R (i.e., when TP1 hits)

# Continuation (fallback if no retest)
USE_CONTINUATION = True

# Time windows (balanced)
BREAKOUT_DEADLINE   = "11:00"   # ET deadline for breakout
RETEST_DEADLINE     = "12:00"   # ET deadline for retest
CONTINUATION_CUTOFF = "12:00"   # ET cutoff for continuation entries

# Filters (balanced for more trades + WR)
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True        # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 50.0
RSI_THRESH_SHORT = 50.0

USE_ADX_FILTER = False
ADX_LEN = 14
ADX_MIN = 18.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# OR width vs daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.10
OR_ATR_MAX = 2.00

# Opening 15m volume quantile
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.30

# Gap filters (relaxed but keep extremes out)
USE_GAP_FILTER = True
GAP_MIN = 0.003                # 0.3%
GAP_MAX = 0.04                 # 4.0%
ALIGN_WITH_GAP_DIR = False     # allow counter-gap trades

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}        # Tue–Thu; only used if USE_DOW_FILTER = True

# Optional index confirmation
USE_SPY_CONFIRM = False
SPY_TICKER = "SPY"

# Risk / exits (if USE_SCALE_OUT=False, these are used)
R_MULTIPLES = [2.0]            # not used when USE_SCALE_OUT=True (kept for compatibility)
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Optional trailing for remainder (off by default)
USE_ATR_TRAIL = False
ATR15_LEN = 14
ATR15_MULT = 2.0

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.4
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# STRESS TEST CONTROLS (OFF by default)
# =============================

RUN_MONTE_CARLO_AFTER = False     # set True to run MC after main backtest
MC_TRIALS = 2000
MC_RANDOM_SEED = 7
MC_SAMPLE_SIZE = None             # None = same length as trades

RUN_SENSITIVITY_GRID = False      # set True to re-run universe on a grid
SENS_SLIPPAGE_LIST = [0.5, 1.0, 2.0]  # bps
SENS_FEE_LIST      = [0.00, 0.25]     # $ per trade
SENS_MAX_TICKERS   = None

# =============================
# Helpers
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    """Return a per-bar session key (one per trading day). Accepts DataFrame or index-like."""
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    """Normalize/pad/truncate the splits to match number of targets."""
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

# Overall Totals (adds Profit Factor & Avg Win/Loss)
def print_overall_totals(trades: pd.DataFrame):
    """Print overall totals across all tickers/trades (with Profit Factor)."""
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return

    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0

    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())

    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())  # positive
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)

    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0  # negative

    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# =============================
# S&P 500 scraping
# =============================

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def sp500_from_wikipedia() -> pd.DataFrame:
    html = requests.get(WIKI_URL, headers={"User-Agent":"Mozilla/5.0"}, timeout=30).text
    table = pd.read_html(html)[0]
    df = table.rename(columns={"Symbol":"SymbolRaw"}).copy()
    df["Symbol"] = df["SymbolRaw"].astype(str).str.strip()
    df["Ticker"] = df["Symbol"].str.replace(r"\.", "-", regex=True)  # BRK.B -> BRK-B
    keep = ["Ticker","Symbol","Security","GICS Sector","GICS Sub-Industry","Headquarters Location"]
    return df[keep]

# =============================
# Data helpers
# =============================

def fetch_intraday(
    ticker: str,
    interval: str = INTERVAL,
    tz: str = TZ,
    period: str = PERIOD,
    use_period: bool = USE_PERIOD
) -> pd.DataFrame:
    intraday_small = interval in {"1m","2m","5m","15m","30m"}
    try:
        if intraday_small and use_period:
            df = yf.download(
                ticker, period=period, interval=interval,
                auto_adjust=False, progress=False, group_by="column", threads=True
            )
        else:
            df = yf.download(
                ticker, period=period, interval=interval,
                auto_adjust=False, progress=False, group_by="column", threads=True
            )
    except Exception as e:
        print(f"[{ticker}] download error: {e}")
        return pd.DataFrame()

    if df is None or df.empty:
        print(f"[{ticker}] no data (interval={interval}, period={period})")
        return pd.DataFrame()

    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) and len(c) else c for c in df.columns]

    # TZ handling
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    df = df.tz_convert(tz)

    # Regular session only & basic cleanup
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith("adj")], errors="ignore")
    df["Ticker"] = ticker
    df = df.sort_index()
    return ensure_unique_columns(df)

# =============================
# Indicators & features
# =============================

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])

    for col in ("High","Low"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")

    if not df.index.is_monotonic_increasing:
        df = df.sort_index()

    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp
        df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff()
    dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)

    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()

    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)

    # Daily EMAs (trend bias)
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean()
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean()

    # Daily ATR
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean()

    # Opening 15m volume + rolling quantile threshold (if Volume exists)
    if "Volume" in df_15.columns:
        tmp = df_15.copy()
        tmp["Session"] = sessionize(tmp)
        open15_vol = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol"] = open15_vol
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
        )

    # Merge ONLY feature columns
    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]

    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

# =============================
# Backtest (per-ticker)
# =============================

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # Opening range + features
    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    # Gap (first 15m vs previous day's close)
    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    # Intraday indicators
    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        # All breakout candidates after the OR bar (use cushion)
        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])  # by timestamp

        trades_this_session = 0
        took_long = False
        took_short = False

        for direction, btime, breakout_row in cands:
            # Respect time window for breakout bar
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short":
                    break
                if took_short and direction == "long":
                    break
            if direction == "long" and took_long:
                continue
            if direction == "short" and took_short:
                continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            # Retest logic
            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                # Continuation fallback
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            # Retest confirm close w/ cushion
            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok:
                    continue

            # Wick filter on entry bar (retest or continuation)
            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            # Filters
            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW:
                continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising):
                        continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling):
                        continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX):
                    continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0):
                    continue

            if USE_SPY_CONFIRM and spy15 is not None and retest_time in spy15.index:
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            # Time windows for the entry bar
            if retest_used and (not _clock_le(retest_time, RETEST_DEADLINE)): continue
            if (not retest_used) and (not _clock_le(retest_time, CONTINUATION_CUTOFF)): continue

            # Entry/stop/risk
            entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
            stop = orl if direction == "long" else orh
            rps = abs(entry - stop)
            if rps <= 1e-12:
                continue

            qty = POSITION_SIZE_DOLLARS / entry
            side_mult = 1 if direction == "long" else -1

            # Targets
            if USE_SCALE_OUT:
                targets_r = TARGETS_R
            else:
                targets_r = r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            # Walk-forward exits
            run = sdf[sdf.index >= retest_time].copy()
            qty_left = qty
            exits = []
            be_armed = (MOVE_STOP_TO_BE_AT_R is not None)
            be_level = entry
            next_tp_idx = 0

            # Leg sizes (normalize to sum=1 across number of targets)
            if USE_SCALE_OUT and len(targets) >= 1:
                splits = _normalize_splits(len(targets), SCALE_SPLIT)
                leg_sizes = [qty * s for s in splits]
            else:
                leg_sizes = [qty]

            for ts, row in run.iterrows():
                hi, lo = row["High"], row["Low"]

                # Arm BE after +X R (even intra-bar) – here MOVE_STOP_TO_BE_AT_R=1.0
                if be_armed and MOVE_STOP_TO_BE_AT_R is not None:
                    if direction == "long" and hi >= entry + (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = max(stop, be_level); be_armed = False
                    elif direction == "short" and lo <= entry - (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = min(stop, be_level); be_armed = False

                # Optional ATR trail on remainder
                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                # Stop first
                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0
                    break

                # Targets (sequential)
                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        next_tp_idx += 1
                        # After first TP hit, lock BE (matches MOVE_STOP_TO_BE_AT_R=1.0)
                        if next_tp_idx == 1 and MOVE_STOP_TO_BE_AT_R is not None:
                            stop = be_level
                        if qty_left <= 1e-9:
                            break

                # If no TP1 by continuation cutoff, tighten to BE
                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            # EOD exit for any leftover
            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(last["Close"], SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits) - fees

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": retest_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            # mark side taken
            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Optional plotting (off by default)
# =============================

def plot_example_days(df_15: pd.DataFrame, trade_log: pd.DataFrame, max_days: int = 0, title_prefix: str = ""):
    if max_days <= 0 or trade_log.empty:
        return
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return
    sessions = trade_log["Session"].unique()[:max_days]
    for ses in sessions:
        sdf = df_15[df_15.index.date == pd.to_datetime(ses).date()]
        if sdf.empty:
            continue
        tr = trade_log[trade_log["Session"] == ses].iloc[0]
        plt.figure(figsize=(11,5))
        plt.plot(sdf.index, sdf["Close"], label="Close")
        plt.axhline(tr["ORH"], linestyle="--", label="ORH")
        plt.axhline(tr["ORL"], linestyle="--", label="ORL")
        plt.scatter([tr["EntryTime"]], [tr["Entry"]], marker="^" if tr["Direction"]=="long" else "v", s=80, label="Entry")
        for tag, tstamp, px, *_ in tr["Exits"]:
            plt.scatter([tstamp], [px], marker="x", s=80, label=f"Exit {tag}")
        plt.title(f"{title_prefix}{tr['Ticker']} {pd.to_datetime(ses).date()} ORB Retest (15m)")
        plt.legend(); plt.tight_layout(); plt.show()

# =============================
# Monte Carlo & Sensitivity (optional)
# =============================

def max_drawdown(equity_curve: np.ndarray) -> float:
    high_water = np.maximum.accumulate(equity_curve)
    drawdowns = high_water - equity_curve
    return float(np.max(drawdowns)) if len(drawdowns) else 0.0

def monte_carlo_bootstrap(trades: pd.DataFrame, trials: int = 2000, sample_size: Optional[int] = None, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0).values
    N = len(pnl)
    if N == 0:
        print("No trades for Monte Carlo.")
        return pd.DataFrame()

    m = sample_size if (sample_size is not None and sample_size > 0) else N
    out = []
    for _ in range(trials):
        sample = pnl[rng.integers(0, N, size=m)]
        tot = float(sample.sum())
        wr = float((sample > 0).mean() * 100.0)
        gp = float(sample[sample > 0].sum())
        gl = float(-sample[sample < 0].sum())
        pf = (gp / gl) if gl > 0 else (float("inf") if gp > 0 else 0.0)
        eq = np.cumsum(sample)
        mdd = max_drawdown(eq)
        out.append((tot, wr, pf, mdd))
    mc = pd.DataFrame(out, columns=["TotPnL","WinRate_%","PF","MaxDD_$"])

    def pct(s, p): return float(np.percentile(s, p))
    print("\n=== Monte Carlo (bootstrap) ===")
    print(f"Trials: {trials} | Sample size: {m}")
    print(f"TotPnL  median: {pct(mc['TotPnL'],50):,.2f} | 10%: {pct(mc['TotPnL'],10):,.2f} | 90%: {pct(mc['TotPnL'],90):,.2f}")
    print(f"WR%     median: {pct(mc['WinRate_%'],50):.2f} | 10%: {pct(mc['WinRate_%'],10):.2f} | 90%: {pct(mc['WinRate_%'],90):.2f}")
    print(f"PF      median: {pct(mc['PF'],50):.3f} | 10%: {pct(mc['PF'],10):.3f} | 90%: {pct(mc['PF'],90):.3f}")
    print(f"MaxDD$  median: {pct(mc['MaxDD_$'],50):,.2f} | 90% worst: {pct(mc['MaxDD_$'],90):,.2f}")
    return mc

def process_universe(tickers: List[str], slippage_bps: float, fees: float, spy15: Optional[pd.DataFrame] = None,
                     autosave_every: int = AUTOSAVE_EVERY, sleep_between: float = SLEEP_BETWEEN_TICKERS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_trades, all_equity = [], []
    processed = 0

    for tk in tickers:
        print(f"\n== {tk} ==")
        raw = fetch_intraday(tk, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
        if raw.empty:
            print("No data.")
            time.sleep(sleep_between); continue

        df15 = ensure_unique_columns(raw.between_time(REG_SESSION_START, REG_SESSION_END))
        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=slippage_bps,
            fees=fees,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=spy15
        )

        if not tlog.empty:
            tlog_out = tlog.copy()
            # Keep qty in Exits for analytics; write qty to CSV too
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = tk
            all_equity.append(eq_out)

        processed += 1
        if autosave_every and processed % autosave_every == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(sleep_between)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def sensitivity_slippage_fee_grid(tickers: List[str], slippages: List[float], fees: List[float],
                                  spy15: Optional[pd.DataFrame]=None, max_tickers: Optional[int]=None) -> pd.DataFrame:a
    if max_tickers is not None:
        tickers = tickers[:max_tickers]
    rows = []
    for bps in slippages:
        for fee in fees:
            print(f"\n=== Sensitivity run: slippage={bps} bps, fee=${fee:.2f} ===")
            trades, _ = process_universe(tickers, slippage_bps=bps, fees=fee, spy15=spy15,
                                         autosave_every=0, sleep_between=0.0)
            pnl = pd.to_numeric(trades.get("PnL_$", pd.Series(dtype=float)), errors="coerce").fillna(0.0)
            wins = pnl > 0
            gp = float(pnl[wins].sum()); gl = float(-pnl[~wins & (pnl<0)].sum())
            pf = (gp / gl) if gl > 0 else (float("inf") if gp > 0 else 0.0)
            row = {
                "slippage_bps": bps,
                "fee_$": fee,
                "Trades": int(len(trades)),
                "TotalPnL_$": float(pnl.sum()),
                "WinRate_%": float(wins.mean()*100.0) if len(pnl) else 0.0,
                "PF": pf
            }
            rows.append(row)
    out = pd.DataFrame(rows).sort_values(["slippage_bps","fee_$"]).reset_index(drop=True)
    out.to_csv("orb_sensitivity_grid.csv", index=False)
    print("\n=== Slippage/Fee Sensitivity Grid ===")
    print(out.to_string(index=False))
    return out

# =============================
# Runner (S&P 500)
# =============================

def run_sp500():
    # Ticker universe
    sp = sp500_from_wikipedia()
    tickers = sp["Ticker"].dropna().unique().tolist()
    print(f"Tickers loaded: {len(tickers)}")

    # Optional SPY confirm data
    spy15 = None
    if USE_SPY_CONFIRM:
        print("Downloading SPY for confirmation...")
        spy_raw = fetch_intraday(SPY_TICKER, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
        if not spy_raw.empty:
            spy_df = compute_opening_range(spy_raw.copy())
            spy_df = add_session_vwap(spy_df)
            spy15 = spy_df.rename(columns={"VWAP":"SPY_VWAP","Close":"SPY_Close"})[["SPY_VWAP","SPY_Close"]]

    # Main pass (with current SLIPPAGE_BPS and FEES_PER_TRADE)
    trades, equity = process_universe(tickers, SLIPPAGE_BPS, FEES_PER_TRADE, spy15=spy15)

    # Overall totals (with Profit Factor)
    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    # Per-ticker summary
    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count', 'sum', 'mean', 'std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        # Add win rate & PF
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda s: (pd.to_numeric(s, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        pos = trades.groupby("Ticker")["PnL_$"].apply(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        neg = trades.groupby("Ticker")["PnL_$"].apply(lambda s: -pd.to_numeric(s, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (pos / neg.replace(0, np.nan)).rename("PF")
        summary = summary.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print("No trades generated with current settings.")

    # ---- Monte Carlo (optional) ----
    if RUN_MONTE_CARLO_AFTER and not trades.empty:
        mc = monte_carlo_bootstrap(trades, trials=MC_TRIALS, sample_size=MC_SAMPLE_SIZE, seed=MC_RANDOM_SEED)
        if not mc.empty:
            mc.to_csv("orb_monte_carlo_trials.csv", index=False)

    # ---- Slippage/Fee sensitivity (optional) ----
    if RUN_SENSITIVITY_GRID:
        sensitivity_slippage_fee_grid(
            tickers, SENS_SLIPPAGE_LIST, SENS_FEE_LIST, spy15=spy15,
            max_tickers=SENS_MAX_TICKERS
        )

if __name__ == "__main__":
    run_sp500()


Tickers loaded: 503

== MMM ==

== AOS ==

== ABT ==

== ABBV ==

== ACN ==

== ADBE ==

== AMD ==

== AES ==

== AFL ==

== A ==

== APD ==

== ABNB ==

== AKAM ==

== ALB ==

== ARE ==

== ALGN ==

== ALLE ==

== LNT ==

== ALL ==

== GOOGL ==

== GOOG ==

== MO ==

== AMZN ==

== AMCR ==

== AEE ==
[autosave] wrote orb_trades_sp500.csv
[autosave] wrote orb_equity_sp500.csv

== AEP ==

== AXP ==

== AIG ==

== AMT ==

== AWK ==

== AMP ==

== AME ==

== AMGN ==

== APH ==

== ADI ==

== AON ==

== APA ==

== APO ==

== AAPL ==

== AMAT ==

== APTV ==

== ACGL ==

== ADM ==

== ANET ==

== AJG ==

== AIZ ==

== T ==

== ATO ==

== ADSK ==

== ADP ==
[autosave] wrote orb_trades_sp500.csv
[autosave] wrote orb_equity_sp500.csv

== AZO ==

== AVB ==

== AVY ==

== AXON ==

== BKR ==

== BALL ==

== BAC ==

== BAX ==

== BDX ==

== BRK-B ==

== BBY ==

== TECH ==

== BIIB ==

== BLK ==

== BX ==

== XYZ ==

== BK ==

== BA ==

== BKNG ==

== BSX ==

== BMY ==

== AVGO ==

== BR ==

== BRO 